In [ ]:
import torch, sys, os, glob
print('python', sys.version)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'gpu{i}', p.name, round(p.total_memory/1024**3, 2), 'GB')
print('input', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else None)
print('draft hits', glob.glob('/kaggle/input/**/config.json', recursive=True)[:8])


In [ ]:
%pip -q install 'transformers==4.57.1' accelerate huggingface_hub datasets
import transformers
print('transformers', transformers.__version__)


In [ ]:
import os, subprocess, sys, glob
from pathlib import Path
hits = glob.glob('/kaggle/input/**/spark-x25-draft-0.5B-base-kd/config.json', recursive=True)
draft = str(Path(hits[0]).parent) if hits else ''
print('DRAFT', draft, flush=True)
repo = 'https://github.com/Priyanshu-5257/knowledgeFlow.git'
dst = '/kaggle/working/repo'
subprocess.check_call(['rm', '-rf', dst])
subprocess.check_call(['git', 'clone', '--depth', '1', repo, dst])
subprocess.check_call(['git', '-C', dst, 'log', '-1', '--oneline'])
cmd = [
    sys.executable, f'{dst}/scripts/distill.py',
    '--stage', 'instruct',
    '--draft-path', draft,
    '--n', '400',
    '--seq-len', '384',
    '--max-new', '128',
    '--steps', '100000',
    '--max-seconds', '27600',
    '--save-every', '1000',
    '--grad-accum', '4',
    '--beta', '0.1',
    '--lr', '1e-5',
    '--freeze-embed-steps', '500',
    '--out', '/kaggle/working/spark-x25-draft-0.5B-instruct-kd',
]
print('RUN', cmd, flush=True)
subprocess.check_call(cmd)
